<a href="https://colab.research.google.com/github/likhithapotluri/AAI_520_Test_Prj/blob/main/AAI520_Investment_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-genai yfinance

In [ ]:
import json
import re


def llm_json(prompt):
    """Call the LLM and parse its reply as JSON."""
    text = llm_call(prompt).strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    return json.loads(text)


def preprocess_news(items, max_chars=400):
    """Step 2: clean text, drop duplicates/empties, assign ids."""
    seen, clean = set(), []
    for item in items:
        title = (item.get("title") or "").strip()
        if not title or title.lower() in seen:
            continue
        seen.add(title.lower())
        summary = re.sub(r"<[^>]+>", "", item.get("summary") or "")
        clean.append({
            "id": len(clean) + 1,
            "title": title,
            "summary": summary.strip()[:max_chars],
            "publisher": item.get("publisher"),
            "date": item.get("date"),
        })
    return clean

In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
print("Key found:", api_key is not None)  # prints True/False, never the key itself

MODEL = "gemini-3.5-flash-lite"

client = genai.Client(api_key=api_key)
response = client.models.generate_content(
    model=MODEL,
    contents="Say hello in one sentence.",
)
print(response.text)

Key found: True
Hello, it's wonderful to meet you!


In [ ]:
import time

MODEL = "gemini-3.5-flash-lite"
_cache = {}


def llm_call(prompt, model=MODEL, retries=3):
    """Send a prompt to Gemini and return the text. Caches repeat prompts."""
    key = (model, prompt)
    if key in _cache:
        return _cache[key]
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=model, contents=prompt
            )
            _cache[key] = response.text
            return response.text
        except Exception as e:
            print(f"LLM call failed (attempt {attempt + 1}): {str(e)[:100]}")
            time.sleep(5 * (attempt + 1))
    raise RuntimeError("LLM call failed after retries")

In [ ]:
import yfinance as yf


def get_price_summary(ticker, period="3mo"):
    """Fetch price history and return a compact summary dict."""
    hist = yf.Ticker(ticker).history(period=period)
    if hist.empty:
        return {"error": f"No price data found for {ticker}"}
    first = hist["Close"].iloc[0]
    last = hist["Close"].iloc[-1]
    return {
        "ticker": ticker,
        "period": period,
        "last_close": round(float(last), 2),
        "change_pct": round(float((last / first - 1) * 100), 1),
        "period_high": round(float(hist["High"].max()), 2),
        "period_low": round(float(hist["Low"].min()), 2),
        "avg_daily_volume": int(hist["Volume"].mean()),
    }


data = get_price_summary("AAPL")
print(data)

{'ticker': 'AAPL', 'period': '3mo', 'last_close': 338.7, 'change_pct': 14.1, 'period_high': 344.27, 'period_low': 273.51, 'avg_daily_volume': 52849375}


In [ ]:
prompt = f"""You are a market analyst. Using ONLY the data below, write a
3-sentence market view. Do not invent any numbers.

Data: {data}"""

print(llm_call(prompt))

Over the past 3-month period, AAPL recorded a last close of 338.7 with a positive change of 14.1%. During this timeframe, the stock fluctuated between a period low of 273.51 and a period high of 344.27. Additionally, the asset maintained an average daily volume of 52,849,375 shares.


In [ ]:
def get_financials(ticker):
    """Fetch key company metrics and the latest quarterly results."""
    t = yf.Ticker(ticker)
    info = t.info or {}
    result = {
        "ticker": ticker,
        "company": info.get("shortName"),
        "sector": info.get("sector"),
        "market_cap": info.get("marketCap"),
        "trailing_pe": info.get("trailingPE"),
        "forward_pe": info.get("forwardPE"),
        "profit_margin": info.get("profitMargins"),
        "revenue_growth": info.get("revenueGrowth"),
        "debt_to_equity": info.get("debtToEquity"),
    }
    try:
        q = t.quarterly_income_stmt
        latest = q.columns[:2]  # two most recent quarters
        result["recent_quarters"] = {
            str(col.date()): {
                "revenue": float(q.loc["Total Revenue", col]),
                "net_income": float(q.loc["Net Income", col]),
            }
            for col in latest
        }
    except Exception as e:
        result["recent_quarters"] = f"unavailable ({type(e).__name__})"
    return result


fin = get_financials("AAPL")
print(fin)

{'ticker': 'AAPL', 'company': 'Apple Inc.', 'sector': 'Technology', 'market_cap': 4944362471424, 'trailing_pe': 38.89667, 'forward_pe': 35.332584, 'profit_margin': 0.27618998, 'revenue_growth': 0.164, 'debt_to_equity': 78.445, 'recent_quarters': {'2026-06-30': {'revenue': 109417000000.0, 'net_income': 29789000000.0}, '2026-03-31': {'revenue': 111184000000.0, 'net_income': 29578000000.0}}}


# Helpers and preprocessing

In [ ]:
def get_news(ticker, limit=5):
    """Fetch recent news items as a list of simple dicts."""
    raw = yf.Ticker(ticker).news or []
    items = []
    for n in raw[:limit]:
        c = n.get("content", n)  # newer format nests data under "content"
        items.append({
            "title": c.get("title"),
            "summary": c.get("summary") or c.get("description"),
            "publisher": (c.get("provider") or {}).get("displayName")
                         or n.get("publisher"),
            "date": c.get("pubDate") or n.get("providerPublishTime"),
        })
    return items


news = get_news("AAPL")
for item in news:
    print(item["title"], "|", item["publisher"], "|", item["date"])

Apple and Google Go Hunting for Crypto Talent, Hint at Stablecoin Push | decrypt | 2026-09-21T16:16:03Z
Why AMD Stock Is Rising Fast Today | Motley Fool | 2026-09-21T15:23:00Z
Is It Too Late To Buy Microsoft Stock After Its Summer Run? | Trefis | 2026-09-21T15:13:48Z
Hugging Face Proves These Two AI Hyperscalers Were Best Protected Against AI Threats | 24/7 Wall St. | 2026-09-21T15:11:30Z
JPMorgan Delivers Bullish Message For Apple Stock Fans | GuruFocus.com | 2026-09-21T15:05:24Z


# Classify, extract, summarize

In [ ]:
def classify_news(ticker, articles):
    """Step 3: label each article."""
    prompt = f"""You are classifying news for the stock {ticker}.
For each article, return a JSON list of objects with exactly these keys:
"id" (int), "relevant" (true/false: is it mainly about {ticker}?),
"category" (one of: earnings, product, analyst_rating, macro, legal, other),
"sentiment" (positive, negative, or neutral for {ticker}).
Return ONLY the JSON list, no other text.

Articles: {json.dumps(articles)}"""
    return llm_json(prompt)


def extract_facts(ticker, articles):
    """Step 4: pull key facts from relevant articles only."""
    prompt = f"""From each article below, extract up to 3 key facts about {ticker}.
Use ONLY information in the text. Do not invent anything.
Return a JSON list of objects: {{"id": int, "facts": [strings]}}.
Return ONLY the JSON list, no other text.

Articles: {json.dumps(articles)}"""
    return llm_json(prompt)


def summarize_news(ticker, classified, facts):
    """Step 5: write the final news summary."""
    prompt = f"""Write a 4-sentence news summary for {ticker} using ONLY the
data below. Mention the overall sentiment and the main themes.
Do not invent facts.

Classifications: {json.dumps(classified)}
Extracted facts: {json.dumps(facts)}"""
    return llm_call(prompt)

# Run the whole chain

In [ ]:
def run_news_chain(ticker):
    raw = get_news(ticker, limit=8)                       # Step 1: ingest
    articles = preprocess_news(raw)                       # Step 2: preprocess
    print(f"Step 2: {len(articles)} clean articles")

    labels = classify_news(ticker, articles)              # Step 3: classify
    print("Step 3: classification")
    for row in labels:
        print("  ", row)

    relevant_ids = {r["id"] for r in labels if r.get("relevant")}
    relevant = [a for a in articles if a["id"] in relevant_ids]
    print(f"Step 3 kept {len(relevant)} of {len(articles)} as relevant")

    facts = extract_facts(ticker, relevant)               # Step 4: extract
    print("Step 4: extracted facts")
    for row in facts:
        print("  ", row)

    summary = summarize_news(ticker, labels, facts)       # Step 5: summarize
    print("\nStep 5: summary\n", summary)
    return {"articles": articles, "labels": labels,
            "facts": facts, "summary": summary}


news_result = run_news_chain("AAPL")

Step 2: 8 clean articles
Step 3: classification
   {'id': 1, 'relevant': False, 'category': 'other', 'sentiment': 'neutral'}
   {'id': 2, 'relevant': True, 'category': 'product', 'sentiment': 'positive'}
   {'id': 3, 'relevant': False, 'category': 'other', 'sentiment': 'neutral'}
   {'id': 4, 'relevant': False, 'category': 'other', 'sentiment': 'neutral'}
   {'id': 5, 'relevant': True, 'category': 'product', 'sentiment': 'positive'}
   {'id': 6, 'relevant': True, 'category': 'analyst_rating', 'sentiment': 'positive'}
   {'id': 7, 'relevant': False, 'category': 'other', 'sentiment': 'neutral'}
   {'id': 8, 'relevant': True, 'category': 'other', 'sentiment': 'positive'}
Step 3 kept 4 of 8 as relevant
Step 4: extracted facts
   {'id': 2, 'facts': ['Apple is hiring for senior roles covering consumer payment products.', 'Apple is hunting for crypto talent and hinting at a stablecoin push.']}
   {'id': 5, 'facts': ["Apple's architecture powers its ambitious bets on Siri, Robotaxi, and beyond

# routing (workflow pattern #2)

## Cell 1: Build the items to route

In [ ]:
ROUTES = ["earnings", "news", "market"]


def build_research_items(ticker, price, fin, news_result):
    """Collect price data, financials and relevant news into one item list."""
    items = [
        {"id": 1, "text": f"Price data: {json.dumps(price)}"},
        {"id": 2, "text": f"Company financials: {json.dumps(fin)}"},
    ]
    relevant_ids = {r["id"] for r in news_result["labels"] if r.get("relevant")}
    for a in news_result["articles"]:
        if a["id"] in relevant_ids:
            items.append({
                "id": len(items) + 1,
                "text": f"News: {a['title']}. {a['summary']}",
            })
    return items

## Cell 2: Router and specialists

In [ ]:
def route_items(ticker, items):
    """Router: the LLM assigns each item to one specialist."""
    prompt = f"""You are a router for a stock research system on {ticker}.
Assign each item to exactly one specialist:
- "earnings": revenue, profit, margins, valuation ratios, earnings results
- "market": price movement, trading volume, analyst ratings, price targets, market conditions
- "news": product launches, strategy, legal or regulatory issues, hiring, partnerships, other company events
Return ONLY a JSON list of objects: {{"id": int, "route": "earnings" or "market" or "news", "reason": short string}}.

Items: {json.dumps(items)}"""
    decisions = llm_json(prompt)
    valid = {d["id"]: d for d in decisions if d.get("route") in ROUTES}
    for item in items:  # fallback if the router skipped or mislabelled an item
        if item["id"] not in valid:
            valid[item["id"]] = {"id": item["id"], "route": "news",
                                 "reason": "fallback: invalid router output"}
    return list(valid.values())


SPECIALISTS = {
    "earnings": "You are an earnings analyst. Focus on revenue, profit, margins, growth and valuation.",
    "market": "You are a market analyst. Focus on price trend, volume, analyst views and market conditions.",
    "news": "You are a news analyst. Focus on company events, products, strategy and risks.",
}


def run_specialist(ticker, route, items):
    prompt = f"""{SPECIALISTS[route]}
Stock: {ticker}. Write 3 sentences using ONLY the items below.
Do not invent numbers. Attribute promotional claims to their source
instead of stating them as fact.

Items: {json.dumps(items)}"""
    return llm_call(prompt)

## Cell 3: Run the routing

In [ ]:
def run_routing(ticker, items):
    decisions = route_items(ticker, items)
    lookup = {i["id"]: i for i in items}
    by_route = {r: [] for r in ROUTES}

    print("Router decisions:")
    for d in decisions:
        print(f"  item {d['id']} -> {d['route']}: {d['reason']}")
        by_route[d["route"]].append(lookup[d["id"]])

    analyses = {}
    for route, group in by_route.items():
        if group:
            analyses[route] = run_specialist(ticker, route, group)
            print(f"\n[{route} analyst]\n{analyses[route]}")
    return decisions, analyses


price = get_price_summary("AAPL")
fin = get_financials("AAPL")
items = build_research_items("AAPL", price, fin, news_result)
decisions, analyses = run_routing("AAPL", items)

Router decisions:
  item 1 -> market: Contains price movement, trading volume, and performance periods.
  item 2 -> earnings: Contains revenue, profit margins, P/E ratios, and quarterly financials.
  item 3 -> news: Discusses company strategy, hiring, and crypto/stablecoin exploration.
  item 4 -> news: Covers AI architecture updates, Siri developments, and company events.
  item 5 -> market: Includes analyst commentary and product demand surges affecting stock views and ratings.
  item 6 -> market: Focuses on investment scenarios, trading near all-time highs, and market momentum.

[earnings analyst]
Apple Inc. posted a profit margin of approximately 27.62% alongside a revenue growth rate of 16.4%. For the quarter ending June 30, 2026, the technology company generated $109,417,000,000 in revenue and $29,789,000,000 in net income. Furthermore, AAPL trades at a trailing price-to-earnings ratio of 38.90 and a forward price-to-earnings ratio of 35.34 within a market capitalization of $4,94

# Pattern 3: Evaluator-optimizer

## Cell 1: Draft, evaluate, optimize

In [16]:
RUBRIC = ["groundedness", "completeness", "balance", "clarity"]


def write_draft(ticker, analyses):
    """Generate: combine specialist analyses into one report."""
    prompt = f"""Write a concise investment research report on {ticker}.
Sections: Summary, Price and Market, Earnings and Financials,
News and Events, Risks and Caveats.
Use ONLY the specialist analyses below. Do not invent facts. If evidence
is thin, say so.

Analyses: {json.dumps(analyses)}"""
    return llm_call(prompt)


def evaluate_report(ticker, report, items):
    """Evaluate: score the report against a rubric using the source items."""
    prompt = f"""You are a strict reviewer of a stock research report on {ticker}.
Compare the report to the SOURCE ITEMS and score each criterion 1-10:
- groundedness: every number and claim is supported by the source items;
  promotional claims are attributed, not stated as fact
- completeness: covers price, fundamentals and news
- balance: presents risks and caveats, not just positives
- clarity: well organised and easy to read
Be strict. Give 10 only if nothing could be improved.
Return ONLY JSON: {{"scores": {{"groundedness": int, "completeness": int,
"balance": int, "clarity": int}}, "feedback": [specific problems to fix]}}

SOURCE ITEMS: {json.dumps(items)}

REPORT: {report}"""
    result = llm_json(prompt)
    scores = result["scores"]
    result["overall"] = round(sum(scores[k] for k in RUBRIC) / len(RUBRIC), 1)
    return result


def optimize_report(ticker, report, feedback, items):
    """Optimize: rewrite the report using the reviewer's feedback."""
    prompt = f"""Revise this research report on {ticker} to fix every problem
in the feedback. Use ONLY the source items. Remove any claim that the
source items do not support.

FEEDBACK: {json.dumps(feedback)}
SOURCE ITEMS: {json.dumps(items)}

REPORT: {report}"""
    return llm_call(prompt)

## Cell 2: The loop

In [17]:
def run_evaluator_optimizer(ticker, analyses, items, threshold=8, max_iter=3):
    report = write_draft(ticker, analyses)
    history = []
    for i in range(1, max_iter + 1):
        review = evaluate_report(ticker, report, items)
        history.append({"iteration": i, "report": report, "review": review})
        print(f"Iteration {i}: scores={review['scores']}  overall={review['overall']}")
        for f in review["feedback"]:
            print("   feedback:", f)
        if review["overall"] >= threshold or i == max_iter:
            break
        report = optimize_report(ticker, report, review["feedback"], items)
    return report, history


final_report, history = run_evaluator_optimizer("AAPL", analyses, items)

print("\n===== FIRST DRAFT =====\n", history[0]["report"])
print("\n===== FINAL REPORT =====\n", final_report)

Iteration 1: scores={'groundedness': 9, 'completeness': 9, 'balance': 7, 'clarity': 9}  overall=8.5
   feedback: The balance section relies heavily on inferred risks rather than direct risks from the source items, though the valuation risk is well-supported by the high P/E ratio.
   feedback: Ensure all claims about AI architecture threat protection accurately reflect the source text without overstating the security claims.

===== FIRST DRAFT =====
 # Investment Research Report: Apple Inc. (AAPL)

### Summary
Apple Inc. (AAPL) exhibits strong earnings momentum and market performance, trading near all-time highs. Supported by surging demand for the upcoming iPhone 18 Pro and strategic moves into artificial intelligence and digital payments, the company maintains robust profitability and high market valuation.

### Price and Market
* **Closing Price:** $338.82
* **Recent Performance:** Shares gained 14.2% over the past three months, trading near period highs of $344.27 and all-time highs